# Calibration Notebook

A toy instrument-calibration workflow. We:
1. Generate synthetic raw observations across three channels.
2. Apply per-channel linear calibration.
3. Compute summary statistics.
4. Plot raw vs calibrated.

Run cells with **Shift+Enter**. Open the Variables panel (top-right of the notebook) to see arrays as you go.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## 1. Generate raw observations

Three channels, 200 samples each, with channel-specific means and a bit of noise.

In [ ]:
n_per_channel = 200
channels = np.repeat([1, 2, 3], n_per_channel)
raw_means = {1: 10.0, 2: 8.5, 3: 5.0}
raw_values = np.concatenate([
    rng.normal(loc=raw_means[ch], scale=0.5, size=n_per_channel)
    for ch in [1, 2, 3]
])

print(f"channels.shape = {channels.shape}")
print(f"raw_values.shape = {raw_values.shape}")
print(f"raw_values[:5] = {raw_values[:5]}")

## 2. Apply calibration

Linear per-channel: `y = slope * x + offset`.

In [ ]:
COEFFICIENTS = {
    1: (1.02, -0.15),
    2: (0.97, 0.04),
    3: (1.00, 0.00),
}

def calibrate(channel: np.ndarray, raw: np.ndarray) -> np.ndarray:
    slope = np.array([COEFFICIENTS[c][0] for c in channel])
    offset = np.array([COEFFICIENTS[c][1] for c in channel])
    return slope * raw + offset

calibrated = calibrate(channels, raw_values)
calibrated[:5]

## 3. Summary statistics per channel

Open the Variables panel and double-click `summary` to view it as a table.

In [ ]:
summary = {}
for ch in [1, 2, 3]:
    mask = channels == ch
    summary[ch] = {
        "n": int(mask.sum()),
        "raw_mean": float(raw_values[mask].mean()),
        "calibrated_mean": float(calibrated[mask].mean()),
        "calibrated_std": float(calibrated[mask].std()),
    }

for ch, s in summary.items():
    print(f"channel {ch}: n={s['n']}, raw_mean={s['raw_mean']:.3f}, "
          f"cal_mean={s['calibrated_mean']:.3f}, cal_std={s['calibrated_std']:.3f}")

## 4. Plot raw vs calibrated

The figure renders inline. Right-click to save as PNG.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ch, color in zip([1, 2, 3], ["C0", "C1", "C2"]):
    mask = channels == ch
    axes[0].hist(raw_values[mask], bins=30, alpha=0.6, color=color, label=f"ch {ch}")
    axes[1].hist(calibrated[mask], bins=30, alpha=0.6, color=color, label=f"ch {ch}")

axes[0].set_title("Raw")
axes[1].set_title("Calibrated")
for ax in axes:
    ax.set_xlabel("value")
    ax.legend()
axes[0].set_ylabel("count")

fig.tight_layout()
plt.show()

## What to try next

- Change a coefficient in `COEFFICIENTS` and re-run the calibration cell. Watch how the plot and summary update — no need to restart the kernel.
- Open the Variables panel. Double-click `calibrated` to view as a table.
- Save and look at the Source Control panel — note the cell-aware diff.
- Cmd+Shift+P → "Jupyter: Convert to Python Script" — see the `# %%` script version.